# 04k | Drills 1: hash maps, two pointers, sliding windows, stacks

Covers **04a–04d**: trading space for speed with dict / set / `Counter`, opposite-direction and same-direction two pointers, fixed- and variable-length windows, stacks and monotonic stacks. Most OA problems 1 and 2 fall into these categories.

**How to use this lesson:** there are no input boxes here; each problem is followed directly by its answer and output. After reading a problem, think first (in your head or on paper): **which pattern is this?** Then read on. After each problem, cover the answer and try to state the approach and its complexity.

Problems on the same page run in order, so functions defined earlier can be reused later.

### Problem 1 · Duplicates within distance k

Does `nums` contain two equal numbers whose indices differ by at most `k`?

`[1, 2, 3, 1], k=3` → `True`; `[1, 2, 3, 1, 2, 3], k=2` → `False`.

```python
def near_duplicate(nums, k):
    last = {}                          # number -> index of its most recent occurrence
    for i, x in enumerate(nums):
        if x in last and i - last[x] <= k:
            return True
        last[x] = i
    return False

print(near_duplicate([1, 2, 3, 1], 3))
print(near_duplicate([1, 2, 3, 1, 2, 3], 2))
print(near_duplicate([], 1))
```

**Result:**

```text
True
False
False
```

> A dict records "where each number last appeared"; you only need to compare with the most recent occurrence. O(n).

### Problem 2 · Intersection of two arrays (with duplicates)

Return the intersection of two lists, where each number appears as many times as the smaller of its two counts.

`[1, 2, 2, 1]` and `[2, 2]` → `[2, 2]`.

```python
from collections import Counter

def intersect(a, b):
    counts = Counter(a)
    out = []
    for x in b:
        if counts[x] > 0:
            out.append(x)
            counts[x] -= 1
    return out

print(intersect([1, 2, 2, 1], [2, 2]))
print(intersect([4, 9, 5], [9, 4, 9, 8, 4]))
```

**Result:**

```text
[2, 2]
[9, 4]
```

> Count a first, then walk through b, decrementing each count as you use it. Looking up a missing key in a `Counter` returns 0 instead of raising an error. O(n + m).

### Problem 3 · Top k most frequent elements

Return the `k` numbers that appear most often in `nums`.

`[1, 1, 1, 2, 2, 3], k=2` → `[1, 2]`.

```python
from collections import Counter

def top_k(nums, k):
    return [x for x, c in Counter(nums).most_common(k)]

top_k([1, 1, 1, 2, 2, 3], 2)
```

**Result:**

```text
[1, 2]
```

> `most_common(k)` returns a list of (element, count) pairs, from most to least frequent. Counting is O(n); to pick the top k, `most_common(k)` uses a heap of size k internally, O(n log k). In an interview it's enough to say "sort by count and take the top k, O(n log n); a size-k heap brings it down to O(n log k)".

### Problem 4 · Longest consecutive sequence

Given an unsorted `nums`, return the length of the longest run of consecutive values, in O(n).

`[100, 4, 200, 1, 3, 2]` → `4` (1, 2, 3, 4).

```python
def longest_consecutive(nums):
    s = set(nums)
    best = 0
    for x in s:
        if x - 1 not in s:             # x starts a run
            length = 1
            while x + length in s:
                length += 1
            best = max(best, length)
    return best

print(longest_consecutive([100, 4, 200, 1, 3, 2]))
print(longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]))
print(longest_consecutive([]))
```

**Result:**

```text
4
9
0
```

> Sorting is O(n log n). With a set, count forward only from starting points "whose predecessor is absent"; each number is counted at most once, O(n) in total.

### Problem 5 · Word pattern

`pattern = "abba"`, `s = "dog cat cat dog"`: is there a one-to-one mapping between letters and words?

```python
def word_pattern(pattern, s):
    words = s.split(" ")
    if len(words) != len(pattern):
        return False
    p2w = {}
    w2p = {}
    for p, w in zip(pattern, words):
        if p in p2w and p2w[p] != w:
            return False
        if w in w2p and w2p[w] != p:
            return False
        p2w[p] = w
        w2p[w] = p
    return True

print(word_pattern("abba", "dog cat cat dog"))
print(word_pattern("abba", "dog cat cat fish"))
print(word_pattern("abba", "dog dog dog dog"))
```

**Result:**

```text
True
False
False
```

> Check **both directions**: if you only check letter → word, `"abba"` and `"dog dog dog dog"` would wrongly return True. O(n).

### Problem 6 · Ransom note

Can you build `note` from the letters in `magazine` (each letter used at most once)?

```python
from collections import Counter

def can_construct(note, magazine):
    have = Counter(magazine)
    for ch, need in Counter(note).items():
        if have[ch] < need:
            return False
    return True

print(can_construct("aa", "aab"), can_construct("aa", "ab"))
```

**Result:**

```text
True False
```

> Count both strings, then compare counts letter by letter. O(n + m).

### Problem 7 · Remove duplicates from a sorted array in place

Remove duplicates from the ascending `nums` in place, return the new length `k`, and put the deduplicated values in the first `k` positions.

```python
def remove_duplicates(nums):
    if not nums:
        return 0
    k = 1                              # nums[:k] is the deduplicated part
    for i in range(1, len(nums)):
        if nums[i] != nums[k - 1]:
            nums[k] = nums[i]
            k += 1
    return k

nums = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]
k = remove_duplicates(nums)
print(k, nums[:k])
```

**Result:**

```text
5 [0, 1, 2, 3, 4]
```

> Same-direction two pointers: `i` reads ahead, `k` points to the next write position. Same template as "Move Zeroes" in 04b. O(n) time, O(1) space.

### Problem 8 · Container with most water

`height[i]` is the height of the i-th vertical line. Pick two lines that, together with the x-axis, form a container. What's the most water it can hold?

`[1, 8, 6, 2, 5, 4, 8, 3, 7]` → `49`.

```python
def max_area(height):
    left, right = 0, len(height) - 1
    best = 0
    while left < right:
        h = min(height[left], height[right])
        best = max(best, h * (right - left))
        if height[left] < height[right]:
            left += 1
        else:
            right -= 1
    return best

max_area([1, 8, 6, 2, 5, 4, 8, 3, 7])
```

**Result:**

```text
49
```

> Opposite-direction two pointers. The water level is set by the **shorter line**; moving the taller one only shrinks the width while the height stays the same or drops, so always move the shorter one. O(n).

### Problem 9 · Is subsequence

Is `s` a subsequence of `t` (appearing in order, not necessarily contiguous)? `"abc"`, `"ahbgdc"` → `True`.

```python
def is_subsequence(s, t):
    i = 0
    for ch in t:
        if i < len(s) and s[i] == ch:
            i += 1
    return i == len(s)

print(is_subsequence("abc", "ahbgdc"), is_subsequence("axc", "ahbgdc"), is_subsequence("", "abc"))
```

**Result:**

```text
True False True
```

> One pointer per string: walk through `t` steadily, and advance `i` each time a character of `s` matches. O(len(t)).

### Problem 10 · 3Sum

Find all unique triplets that sum to 0.

`[-1, 0, 1, 2, -1, -4]` → `[[-1, -1, 2], [-1, 0, 1]]`.

```python
def three_sum(nums):
    nums = sorted(nums)
    out = []
    for i in range(len(nums) - 2):
        if i == 0 or nums[i] != nums[i - 1]:    # handle each first number only once
            left, right = i + 1, len(nums) - 1
            while left < right:
                s = nums[i] + nums[left] + nums[right]
                if s < 0:
                    left += 1
                elif s > 0:
                    right -= 1
                else:
                    out.append([nums[i], nums[left], nums[right]])
                    left += 1
                    while left < right and nums[left] == nums[left - 1]:
                        left += 1      # skip repeated second numbers
    return out

print(three_sum([-1, 0, 1, 2, -1, -4]))
print(three_sum([0, 0, 0, 0]))
```

**Result:**

```text
[[-1, -1, 2], [-1, 0, 1]]
[[0, 0, 0]]
```

> Sort first and fix the first number; what remains is 04b's "find two numbers in a sorted array". Deduplicate by "skip if equal to the previous one". O(n²).

### Problem 11 · Palindrome after deleting at most one character

Can `s` become a palindrome after deleting at most one character? `"abca"` → `True` (delete c).

```python
def valid_palindrome(s):
    def is_pal(i, j):
        while i < j:
            if s[i] != s[j]:
                return False
            i += 1
            j -= 1
        return True

    left, right = 0, len(s) - 1
    while left < right:
        if s[left] != s[right]:
            # the two ends differ: try deleting the left one or the right one
            return is_pal(left + 1, right) or is_pal(left, right - 1)
        left += 1
        right -= 1
    return True

print(valid_palindrome("aba"), valid_palindrome("abca"), valid_palindrome("abc"))
```

**Result:**

```text
True True False
```

> Opposite-direction two pointers. At the first mismatch there are only two possible deletions; check the remainder once for each. O(n).

### Problem 12 · Longest substring with at most two distinct characters

Return the length of the longest substring of `s` containing at most two distinct characters. `"eceba"` → `3` (`"ece"`).

```python
def longest_two_kinds(s):
    counts = {}
    kinds = 0                          # number of distinct characters in the window
    left = 0
    best = 0
    for right, ch in enumerate(s):
        if counts.get(ch, 0) == 0:
            kinds += 1
        counts[ch] = counts.get(ch, 0) + 1
        while kinds > 2:
            out_ch = s[left]
            counts[out_ch] -= 1
            if counts[out_ch] == 0:
                kinds -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

print(longest_two_kinds("eceba"), longest_two_kinds("ccaabbb"), longest_two_kinds(""))
```

**Result:**

```text
3 5 0
```

> Variable-window template. Count the window's characters in a dict and track the number of distinct ones in `kinds`: add one when a count goes from 0 to 1, subtract one when it drops back to 0. O(n).

### Problem 13 · Find all anagram positions

Return the starting indices of all substrings of `s` that are anagrams of `p`.

`s = "cbaebabacd"`, `p = "abc"` → `[0, 6]`.

```python
from collections import Counter

def find_anagrams(s, p):
    k = len(p)
    need = Counter(p)
    window = Counter(s[:k])

    def same():
        # the window is as long as p, so if every letter of p has the right count, they match
        for ch, c in need.items():
            if window[ch] != c:
                return False
        return True

    out = []
    if same():
        out.append(0)
    for right in range(k, len(s)):
        window[s[right]] += 1          # the character coming in
        window[s[right - k]] -= 1      # the character going out
        if same():
            out.append(right - k + 1)
    return out

print(find_anagrams("cbaebabacd", "abc"))
print(find_anagrams("abab", "ab"))
```

**Result:**

```text
[0, 6]
[0, 1, 2]
```

> Fixed-length window: one character in on the right, one out on the left, then compare the two counts. The number of distinct letters in p is bounded (by the alphabet size), so each comparison is O(1), O(n) in total.
>
> Why not just write `window == need`? When a letter leaves the window its count becomes 0, but the key stays. Since Python 3.10, `Counter` comparison treats 0 as missing; in 3.9 and earlier it doesn't, so `{'a': 1, 'e': 0}` and `{'a': 1}` compare unequal. Comparing p's letters one by one works on every version.

### Problem 14 · Number of subarrays with product less than k

`nums` contains only positive integers. How many contiguous subarrays have a product strictly less than `k`?

`[10, 5, 2, 6], k=100` → `8`.

```python
def count_product_less(nums, k):
    if k <= 1:
        return 0
    prod = 1
    left = 0
    count = 0
    for right, x in enumerate(nums):
        prod *= x
        while prod >= k:
            prod //= nums[left]
            left += 1
        count += right - left + 1      # number of valid subarrays ending at right
    return count

print(count_product_less([10, 5, 2, 6], 100), count_product_less([1, 2, 3], 0))
```

**Result:**

```text
8 0
```

> Variable window. When the window `[left, right]` is valid, there are `right - left + 1` subarrays ending at `right`; add them all. O(n). `prod *= x` is shorthand for `prod = prod * x`, and `prod //= nums[left]` works the same way (just like `+=`).

### Problem 15 · Longest repeating character replacement

If you may change at most `k` characters to other letters, how long is the longest substring "made of a single letter" you can get?

`"AABABBA", k=1` → `4`.

```python
def character_replacement(s, k):
    counts = {}
    left = 0
    most = 0                           # count of the most frequent letter in the window
    best = 0
    for right, ch in enumerate(s):
        counts[ch] = counts.get(ch, 0) + 1
        most = max(most, counts[ch])
        while (right - left + 1) - most > k:   # more than k letters would need replacing
            counts[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

print(character_replacement("ABAB", 2), character_replacement("AABABBA", 1))
```

**Result:**

```text
4 4
```

> Window length minus the count of the most frequent letter is the number of letters to replace; the window is valid if that's at most k. O(n). Not decreasing `most` when letters leave on the left is still correct: the answer can only grow when a larger `most` appears, and while `most` is stale the window length stays at most `most + k`, a length already achieved, so `best` never exceeds the true answer.

### Problem 16 · Evaluate reverse Polish notation

Evaluate the postfix expression `["2", "1", "+", "3", "*"]` (that is, (2 + 1) × 3). Division truncates toward zero.

```python
def eval_rpn(tokens):
    stack = []
    for t in tokens:
        if t in ("+", "-", "*", "/"):
            b = stack.pop()            # the one popped second is the left operand
            a = stack.pop()
            if t == "+":
                stack.append(a + b)
            elif t == "-":
                stack.append(a - b)
            elif t == "*":
                stack.append(a * b)
            else:
                stack.append(int(a / b))
        else:
            stack.append(int(t))
    return stack[0]

print(eval_rpn(["2", "1", "+", "3", "*"]))
print(eval_rpn(["4", "13", "5", "/", "+"]))
print(eval_rpn(["6", "-4", "/"]))
```

**Result:**

```text
9
6
-1
```

> Push numbers; on an operator, pop two, compute, and push the result. Mind the order: the first one popped is `b`. `int(a / b)` truncates toward zero, whereas `-6 // 4` gives -2, which is wrong. O(n).

### Problem 17 · Min stack

Design a stack that supports `push`, `pop`, and `top`, and can also return the current minimum in O(1).

```python
class MinStack:
    def __init__(self):
        self.stack = []
        self.mins = []                 # mins[-1] is always the current minimum

    def push(self, x):
        self.stack.append(x)
        if not self.mins or x <= self.mins[-1]:
            self.mins.append(x)
        else:
            self.mins.append(self.mins[-1])

    def pop(self):
        self.mins.pop()
        return self.stack.pop()

    def top(self):
        return self.stack[-1]

    def get_min(self):
        return self.mins[-1]

s = MinStack()
s.push(3); s.push(1); s.push(2)
print(s.get_min(), s.top())
s.pop(); s.pop()
print(s.get_min(), s.top())
```

**Result:**

```text
1 2
3 3
```

> Keep a second stack, in sync with the main one, storing "the minimum up to this level". Every operation is O(1).
>
> Besides `__init__`, a class can define other functions (called **methods**). Their first parameter is also `self`, and they reach the object's own data via `self.stack`; you call them as `s.push(3)` without passing `self`. The semicolons in `s.push(3); s.push(1)` just put several statements on one line.

### Problem 18 · Decode string

`"3[a2[c]]"` → `"accaccacc"`: `k[...]` means repeat the bracketed content k times; brackets can be nested.

```python
def decode_string(s):
    stack = []                         # stores (string before the bracket, repeat count)
    cur = ""
    num = 0
    for ch in s:
        if ch in "0123456789":
            num = num * 10 + int(ch)
        elif ch == "[":
            stack.append((cur, num))
            cur = ""
            num = 0
        elif ch == "]":
            prev, k = stack.pop()
            cur = prev + cur * k
        else:
            cur += ch
    return cur

print(decode_string("3[a]2[bc]"))
print(decode_string("3[a2[c]]"))
print(decode_string("2[abc]3[cd]ef"))
```

**Result:**

```text
aaabcbc
accaccacc
abcabccdcdcdef
```

> On `[`, push "the string built so far" and "the count" onto the stack and start fresh; on `]`, pop them and stitch the result back together. The stack is as deep as the nesting. `num * 10 + int(ch)` handles numbers with two or more digits. `cur * k` repeats a string k times (`"ab" * 3` gives `"ababab"`, just like multiplying a list by an integer).

### Problem 19 · Simplify path

Simplify the Unix path `"/a/./b/../../c/"` to `"/c"` (`.` is the current directory, `..` is the parent).

```python
def simplify_path(path):
    stack = []
    for part in path.split("/"):
        if part == "..":
            if stack:
                stack.pop()
        elif part != "" and part != ".":
            stack.append(part)
    return "/" + "/".join(stack)

print(simplify_path("/a/./b/../../c/"))
print(simplify_path("/../"))
print(simplify_path("/home//foo/"))
```

**Result:**

```text
/c
/
/home/foo
```

> Split on `/`, push ordinary names, pop on `..`. Finally join back with `"/".join`. O(n).

### Problem 20 · Implement a queue with two stacks

Implement a first-in, first-out queue using only stacks (list `append` / `pop`).

```python
class MyQueue:
    def __init__(self):
        self.inbox = []
        self.outbox = []

    def push(self, x):
        self.inbox.append(x)

    def pop(self):
        if not self.outbox:
            while self.inbox:              # pouring it over reverses the order
                self.outbox.append(self.inbox.pop())
        return self.outbox.pop()

q = MyQueue()
q.push(1); q.push(2); q.push(3)
print(q.pop(), q.pop())
q.push(4)
print(q.pop(), q.pop())
```

**Result:**

```text
1 2
3 4
```

> Enqueue into `inbox`; on dequeue, pour all of `inbox` over only when `outbox` is empty. Each element is poured at most once, so each operation is O(1) amortized. In real code, just use `deque` for a queue.

## Pattern recap

| Pattern | When to think of it | Key code |
|---|---|---|
| Hash map | Seen it before? How many times? Where? | `last[x] = i`, `Counter(xs)`, `most_common(k)`, `set` |
| Bidirectional mapping | One-to-one correspondence | Two dicts, check both directions |
| Opposite-direction two pointers | Sorted arrays, palindromes, comparing both ends | `left, right = 0, len(xs) - 1`, `while left < right` |
| Same-direction two pointers | In-place removal, subsequences | One reads, one writes |
| Sort + two pointers | 3Sum | Fix one, find the other two with opposite-direction pointers; skip duplicates |
| Fixed window | Substrings of length k | One in on the right, one out on the left |
| Variable window | Longest / shortest / count satisfying a condition | `for right`, `while invalid: left += 1` |
| Stack | Brackets, expressions, nesting, paths | `stack.append`, `stack.pop()`, `stack[-1]` |
| Auxiliary stack | O(1) minimum | Store "the minimum up to here" in sync |